In [ ]:
# https://github.com/datasets/nyse-other-listings/blob/main/data/nyse-listed.csv
# https://github.com/datasets/nyse-other-listings/blob/main/data/other-listed.csv
# https://github.com/abbadata/stock-tickers/blob/main/data/all.csv

# https://www.nbim.no/en/responsible-investment/ethical-exclusions/exclusion-of-companies/

import csv

caps_rough = {}
names = {}

with open('all.csv', newline='', encoding='utf-8') as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        name, cap = row[0].strip(), row[3]
        if cap == "n/a":
            continue

        if cap[0] != "$":
            raise Exception("bad data " + row[0] + " " + row[3])

        if cap[-1] == "M":
            cap_val = float(cap[1:-1]) * 1_000_000
        elif cap[-1] == "B":
            cap_val = float(cap[1:-1]) * 1_000_000_000
        else:
            cap_val = float(cap[1:])
        
        caps_rough[name] = cap_val

for name in ['nyse-listed.csv', 'other-listed.csv']:
    with open(name, newline='', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile)

        for row in reader:
            names[row[0]] = row[1]

caps_list = [(k, v) for k, v in caps_rough.items()]
caps_list.sort(key=lambda x:x[1], reverse=True)

with open("full.txt", "w") as f:
    for k, v in caps_list:
        f.write(k + ", " + names.get(k, k)[0:200] + "\n")

In [1]:
import re

nesg_tickers = {
    "fastfood": "MCD · YUM · YUMC · QSR · DPZ · WEN · SBUX · CMG · PZZA · JACK · DRI · BLMN · BJRI   · CAKE · DPZ  · WING · TXRH · DENN  · LOCO · SG $KO (Coca-Cola), $PEP (PepsiCo), $MCD, $YUM (k f c/Taco Bell), $QSR (Burger King), **MCD** (McDonald’s) • **KO** (Coca-Cola) • **YUM** (kfc / Taco Bell / Pizza Hut) • **YUMC** (kfc China et al.) • **QSR** (Burger King / Tim Hortons / Popeyes) • **CMG** (Chipotle) • **HSY** (Hershey) • **SJM** (Smucker’s) • **CAG** (ConAgra)",
    "tobacco": "MO · PM · BTI · UVV · $MO (Altria), $PM (Philip Morris), $BTI, **MO** (Altria / Marlboro) • **PM** (Philip Morris Int’l) • **BTI** (British American Tobacco) • **RLX** (RLX Technology – e-cigs in China) • **TPB** (Turning Point Brands – pipe/cigar & vape)",
    "alcohol": "BUD · DEO · TAP · STZ · SAM · CCU · ABEV · FMX ·  **BUD** (AB InBev) • **DEO** (Diageo) • **STZ** (Constellation Brands) • **TAP / TAP.A** (Molson Coors) • **SAM** (Boston Beer) • **ABEV** (Ambev) • **CCU** (Cervecerías Unidas) • **FMX** ",
    "gambling": "LVS · WYNN · MGM · CZR · DKNG · PENN · BYD · BALY · RSI ·  · CHDN ·  · VICI · GLPI $MGM, $WYNN, or online plays like $DKNG, **LVS** (Las Vegas Sands) • **MGM** (MGM Resorts) • **BYD** (Boyd Gaming) • **** (Bally’s) • **RSI** (Rush Street Interactive) • **SGHC** (Super Group – Betway) • **EVRI** (Everi – slot & fintech for casinos) • **AGS** (PlayAGS – slot machines) ",
    "weed": "TLRY · CGC · ACB ·  · SNDL · VFF · GRWG · HYFM · IIPR · OGI · VFF ·  **IIPR** ",
    "hft": "VIRT · CME · CBOE · NDAQ · ICE · SCHW · IBKR · MKTX · CBOE ·  · TTD **SCHW** (Charles Schwab) **IBKR** **HOOD** ",
    "luxury": "TPR (Coach, Kate Spade) · CPRI (Michael Kors, Versace) · RL · SIG · FOSL · SKX · GES · PVH · KSS (mids-tier “luxury” private labels) TPR CPRI RL MOV SIG GES ANF BIRK EL"
}

sizes = {
    "fastfood": 645,
    "tobacco": 886,
    "alcohol": 1762,
    "gambling": 661,
    "weed": 32,
    "hft": 10,
    "luxury": 264
}
sizes_sum = sum(e for e in sizes.values())
sizes = {k: v / sizes_sum for k, v in sizes.items()}

nesg_tickers = {industry: set(re.findall(r"[A-Z]{2,7}", text)) for industry, text in nesg_tickers.items()}

industry_by_company = {}

for industry, companies in nesg_tickers.items():
    for c in companies:
        industry_by_company[c] = industry

tickers = set()

for add in nesg_tickers.values():
	tickers |= add

len(tickers)

86

In [2]:
import datetime as dt

ts = int(dt.datetime(2025, 1, 1, tzinfo=dt.timezone.utc).timestamp())
print(ts)  # 946684800


1735689600


data = {};

for (let ticker of tickers) {
    if (data[ticker]) continue;
    data[ticker] = await fetch(`https://query2.finance.yahoo.com/v8/finance/chart/${ticker}?period1=946684800&period2=1735689600&interval=3mo`).then(res=>res.json());
}

document.body.innerText = JSON.stringify(data);

In [3]:
from tqdm import tqdm
import requests
import time
import json

API_KEY = "***REMOVED***"

with open("state.json", "r") as f:
    state = json.load(f)

for t in tqdm([t for t in tickers if t not in state["tickers"]]):
	time.sleep(12)
	state["tickers"][t] = requests.get(f"https://api.polygon.io/v3/reference/tickers/{t}?apiKey={API_KEY}").json()

with open("state.json", "w") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)

0it [00:00, ?it/s]


In [4]:
stock_data = {name: state["tickers"][name]["results"] for name in tickers if "results" in state["tickers"][name]}
stock_data = {name: {"cap": value["market_cap"], "name": value["name"]} for name, value in stock_data.items() if "market_cap" in value}
total_cap = sum(v["cap"] for v in stock_data.values())

In [5]:
market_cap_shares = {k: sum(stock_data[e]["cap"] for e in v) / total_cap for k, v in nesg_tickers.items()}

industry_weights = {k: (v + sizes[k] ) / 2 for k, v in market_cap_shares.items()}

ind_scale = {k: industry_weights[k] / v for k, v in market_cap_shares.items()}

{k: f"{100 * v:.2f}%" for k, v in industry_weights.items()}

{'fastfood': '27.66%',
 'tobacco': '19.48%',
 'alcohol': '26.66%',
 'gambling': '10.31%',
 'weed': '0.43%',
 'hft': '10.68%',
 'luxury': '4.77%'}

In [7]:
companies = [(name, min(0.0385 * total_cap, data["cap"] * ind_scale[industry_by_company[name]])) for name, data in stock_data.items()]
companies.sort(key=lambda x: x[1], reverse=True)

total = sum(c[1] for c in companies)

cum = 0
i = 0

invest_usd = 6000

portfolio = {}

for name, cap in companies:
    cap /= total
    cum += cap
    i += 1
    portfolio[name] = invest_usd * cap
    print(f"{i} {name} {invest_usd * cap:.0f} {cum:.4f} " + state["tickers"][name]["results"]["name"])

1 BUD 299 0.0499 Anheuser-Busch INBEV SA/NV
2 BTI 299 0.0998 British American Tobacco p.l.c. American Depositary Shares, American Depositary Shares, each representing one Ordinary Share
3 DEO 299 0.1497 Diageo plc
4 MCD 299 0.1996 McDonald's Corporation
5 KO 299 0.2495 Coca-Cola Company
6 PEP 299 0.2995 PepsiCo, Inc.
7 MO 299 0.3494 Altria Group, Inc.
8 PM 299 0.3993 Philip Morris International Inc.
9 ABEV 244 0.4400 AMBEV S.A.
10 FMX 237 0.4795 FOMENTO ECONOMICO MEXICANO, S.A.B. DE C.V.
11 SCHW 233 0.5183 The Charles Schwab Corporation
12 STZ 199 0.5514 Constellation Brands, Inc.
13 VICI 195 0.5839 VICI Properties Inc. Common Stock
14 SBUX 192 0.6158 Starbucks Corp
15 LVS 169 0.6439 Las Vegas Sands Corp.
16 CME 152 0.6693 CME Group Inc.
17 ICE 150 0.6943 Intercontinental Exchange  Inc.
18 CMG 133 0.7165 Chipotle Mexican Grill, Inc.
19 EL 98 0.7328 The Estee Lauder Companies Inc. Class A
20 DKNG 97 0.7490 DraftKings Inc. Class A Common Stock
21 HOOD 87 0.7636 Robinhood Markets, Inc. Cl

In [9]:
total_apy_weighted, total_share = 0, 0

for stock_name, share in portfolio.items():
    stock_data = state["prices"][stock_name]["chart"]["result"][0]
    
    if "timestamp" not in stock_data:
        print(stock_name)
        continue

    timestamps = stock_data["timestamp"]
    prices = stock_data["indicators"]["quote"][0]["open"]

    years = (timestamps[-1] - timestamps[0]) / 365 / 24 / 3600
    ret = prices[-1] / prices[0]

    APY = ret ** (1 / years)

    total_apy_weighted += APY * share
    total_share += share

    
total_apy_weighted / total_share

BALY


1.087380896095192

In [10]:
(6038 / 1452) ** (1 / 25)

1.0586613541620369